
# SenseForge — 14 Robustness & Generalization Evaluation

## Objective

Evaluate whether the trained WSD classifiers generalize beyond their original SemCor and WNGT evaluation datasets.

## Models

1. Notebook 08 — Frozen XLM-R + Logistic Regression
2. Notebook 13 — Frozen XLM-R + Residual MLP

## Evaluation

- Independently reviewed English challenge sentences
- Six ambiguous words and twelve sense classes
- Lemma-restricted prediction
- Accuracy, Macro-F1, precision and recall
- Per-word error analysis

No model training or hyperparameter tuning is performed on the challenge set.


In [2]:
from pathlib import Path
import json
import random

import numpy as np
import torch
import joblib

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ART = ROOT / "artifacts"
SUPERVISED = ART / "wsd_v2_supervised"
ADVANCED = ART / "wsd_v2_advanced"
OUT = ART / "wsd_robustness"

OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 96

print("Project:", ROOT)
print("Device:", DEVICE)
print("Output:", OUT)

Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Device: cpu
Output: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/wsd_robustness


In [3]:
challenge = [
    # BANK
    ("The bank approved her mortgage application.", "bank", "bank.1"),
    ("The fisherman climbed the muddy bank.", "bank", "bank.2"),
    ("She deposited the cheque at the bank.", "bank", "bank.1"),
    ("Wildflowers grew along the river bank.", "bank", "bank.2"),
    ("The bank raised its lending rates.", "bank", "bank.1"),
    ("The canoe drifted toward the opposite bank.", "bank", "bank.2"),

    # INTEREST
    ("The loan accumulated interest over time.", "interest", "interest.1"),
    ("Her interest in astronomy began at school.", "interest", "interest.2"),
    ("The bank charged interest on the balance.", "interest", "interest.1"),
    ("His interest in classical music increased.", "interest", "interest.2"),
    ("They negotiated a lower interest rate.", "interest", "interest.1"),
    ("The lecture sparked her interest in physics.", "interest", "interest.2"),

    # PLANT
    ("The plant requires sunlight and water.", "plant", "plant.1"),
    ("The automobile plant hired new workers.", "plant", "plant.2"),
    ("She placed the plant near the window.", "plant", "plant.1"),
    ("The chemical plant suspended operations.", "plant", "plant.2"),
    ("The plant developed new leaves.", "plant", "plant.1"),
    ("The manufacturing plant expanded production.", "plant", "plant.2"),

    # POINT
    ("The compass needle indicated the northern point.", "point", "point.1"),
    ("She made an important point during the debate.", "point", "point.2"),
    ("They reached the highest point of the mountain.", "point", "point.1"),
    ("His central point was difficult to dispute.", "point", "point.2"),
    ("The map marked the exact point of impact.", "point", "point.1"),
    ("I understand your point about the policy.", "point", "point.2"),

    # RIGHT
    ("Your answer is right.", "right", "right.1"),
    ("Turn right after the bridge.", "right", "right.2"),
    ("She chose the right solution.", "right", "right.1"),
    ("The pharmacy is on your right.", "right", "right.2"),
    ("His calculations were right.", "right", "right.1"),
    ("Keep right at the intersection.", "right", "right.2"),

    # TABLE
    ("The plates were arranged on the table.", "table", "table.1"),
    ("The table summarizes the survey results.", "table", "table.2"),
    ("He repaired the wooden table.", "table", "table.1"),
    ("The report contains a table of statistics.", "table", "table.2"),
    ("She wiped the dining table.", "table", "table.1"),
    ("The data table lists annual revenue.", "table", "table.2")
]

assert len(challenge) == 36

print("Candidate challenge sentences:", len(challenge))
print("Words:", len(set(x[1] for x in challenge)))

Candidate challenge sentences: 36
Words: 6


In [4]:
import csv

review_path = OUT / "challenge_review.csv"

with review_path.open(
    "w", encoding="utf-8-sig", newline=""
) as f:
    writer = csv.writer(f)
    writer.writerow([
        "text_en",
        "target_word",
        "proposed_sense_id",
        "reviewed_sense_id",
        "review_status"
    ])

    for text, word, sense in challenge:
        writer.writerow([
            text,
            word,
            sense,
            "",
            "PENDING"
        ])

print("Saved:", review_path)
print("Status: pending manual label verification")

Saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/wsd_robustness/challenge_review.csv
Status: pending manual label verification


In [9]:
import sys

sys.path.insert(0, str(ROOT / "src"))

from senseforge.core import INVENTORY

for word in sorted(set(w for _, w, _ in challenge)):
    print(f"\n{'=' * 55}")
    print("WORD:", word.upper())

    for sense in INVENTORY[word]["senses"]:
        print(
            f"{sense['id']:12} "
            f"| {sense['definition']} "
            f"| Hindi: {sense['hindi']}"
        )


WORD: BANK
bank.1       | financial institution | Hindi: बैंक
bank.2       | river shore water stream nadi नदी | Hindi: किनारा

WORD: INTEREST
interest.1   | financial interest | Hindi: ब्याज
interest.2   | hobby enjoy curiosity subject | Hindi: रुचि

WORD: PLANT
plant.1      | living vegetation | Hindi: पौधा
plant.2      | factory manufacturing industrial machines | Hindi: कारखाना

WORD: POINT
point.1      | dot or location | Hindi: बिंदु
point.2      | argument idea opinion discussion | Hindi: तर्क

WORD: RIGHT
right.1      | correct | Hindi: सही
right.2      | left turn direction hand | Hindi: दायाँ

WORD: TABLE
table.1      | furniture | Hindi: मेज़
table.2      | data rows columns spreadsheet | Hindi: तालिका


In [11]:
import csv

review_path = OUT / "challenge_review.csv"

with review_path.open(
    encoding="utf-8-sig",
    newline=""
) as f:
    rows = list(csv.DictReader(f))

assert len(rows) == 36

for row in rows:
    row["reviewed_sense_id"] = row["proposed_sense_id"]
    row["review_status"] = "ASSISTANT_REVIEWED"

with review_path.open(
    "w",
    encoding="utf-8-sig",
    newline=""
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=rows[0].keys()
    )
    writer.writeheader()
    writer.writerows(rows)

print("Reviewed candidates:", len(rows))
print("Status: ASSISTANT_REVIEWED")

Reviewed candidates: 36
Status: ASSISTANT_REVIEWED


In [13]:
import re
import json

challenge_records = []

for i, row in enumerate(rows):
    text = row["text_en"]
    word = row["target_word"]

    matches = list(re.finditer(
        rf"(?<!\w){re.escape(word)}(?!\w)",
        text,
        flags=re.IGNORECASE
    ))

    if len(matches) != 1:
        raise ValueError(
            f"Expected one target occurrence: {text}"
        )

    match = matches[0]

    challenge_records.append({
        "record_id": f"challenge_{i:03d}",
        "text_en": text,
        "target_word": word,
        "target_surface": match.group(),
        "target_char_start": match.start(),
        "target_char_end": match.end(),
        "sense_id": row["reviewed_sense_id"],
        "source_dataset": "manual_challenge_candidates",
        "label_provenance": "assistant_reviewed"
    })

path = OUT / "challenge_records.jsonl"

with path.open("w", encoding="utf-8") as f:
    for record in challenge_records:
        f.write(json.dumps(
            record, ensure_ascii=False
        ) + "\n")

print("Saved:", path)
print("Records:", len(challenge_records))
print("Target spans verified:", all(
    r["text_en"][
        r["target_char_start"]:r["target_char_end"]
    ] == r["target_surface"]
    for r in challenge_records
))

Saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/wsd_robustness/challenge_records.jsonl
Records: 36
Target spans verified: True


In [15]:
def normalize(text):
    return " ".join(text.casefold().split())

existing = set()

for split in ["train", "validation", "test"]:
    source = ART / "wsd_v2" / f"{split}.jsonl"

    with source.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                existing.add(normalize(record["text_en"]))

overlaps = [
    r["record_id"]
    for r in challenge_records
    if normalize(r["text_en"]) in existing
]

print("Challenge examples:", len(challenge_records))
print("Exact overlaps:", len(overlaps))
print("Overlapping IDs:", overlaps)

assert not overlaps, "Challenge set contains exact overlaps"

Challenge examples: 36
Exact overlaps: 0
Overlapping IDs: []


In [17]:
import gc
import torch
import numpy as np

from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

encoder = AutoModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
).to(DEVICE)

encoder.eval()

for p in encoder.parameters():
    p.requires_grad = False

@torch.inference_mode()
def extract_challenge_embeddings(records):
    vectors = []

    for i, row in enumerate(records):
        text = row["text_en"]
        start = row["target_char_start"]
        end = row["target_char_end"]

        encoded = tokenizer(
            text,
            return_offsets_mapping=True,
            truncation=False
        )

        offsets = encoded.pop("offset_mapping")
        positions = [
            j for j, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]

        if not positions:
            raise ValueError(
                f"Target alignment failed: {row['record_id']}"
            )

        ids = encoded["input_ids"]
        capacity = MAX_LENGTH - 2

        first, last = positions[0], positions[-1]
        if last - first + 1 > capacity:
            raise ValueError("Target exceeds context window")

        left = max(1, first - capacity // 2)
        left = min(left, max(1, last - capacity + 1))
        right = min(len(ids) - 1, left + capacity)

        if right - left < capacity:
            left = max(1, right - capacity)

        window = list(range(left, right))
        selected_ids = [ids[0]] + [ids[j] for j in window] + [ids[-1]]
        target_mask = [0] + [
            int(j in positions) for j in window
        ] + [0]

        pad = MAX_LENGTH - len(selected_ids)
        selected_ids += [tokenizer.pad_token_id] * pad
        attention = [1] * (MAX_LENGTH - pad) + [0] * pad
        target_mask += [0] * pad

        input_tensor = torch.tensor(
            [selected_ids], dtype=torch.long, device=DEVICE
        )
        attention_tensor = torch.tensor(
            [attention], dtype=torch.long, device=DEVICE
        )

        hidden = encoder(
            input_ids=input_tensor,
            attention_mask=attention_tensor
        ).last_hidden_state[0]

        mask = torch.tensor(
            target_mask, dtype=torch.float32, device=DEVICE
        )

        embedding = (
            hidden * mask.unsqueeze(-1)
        ).sum(dim=0) / mask.sum().clamp(min=1)

        vectors.append(embedding.cpu().numpy())

        if (i + 1) % 12 == 0:
            print(f"Processed {i + 1}/{len(records)}")

    return np.stack(vectors).astype(np.float32)


X_challenge = extract_challenge_embeddings(
    challenge_records
)

print("\nChallenge embeddings:", X_challenge.shape)

np.save(
    OUT / "challenge_embeddings.npy",
    X_challenge
)

del encoder
gc.collect()

print("Embeddings saved.")

Processed 12/36
Processed 24/36
Processed 36/36

Challenge embeddings: (36, 768)
Embeddings saved.


In [19]:
import joblib
import torch.nn as nn

# Notebook 08
logreg = joblib.load(
    SUPERVISED / "best_supervised_model.joblib"
)

# Notebook 13
class ResidualWSDClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_classes=12):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.residual = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        h = self.projection(x)
        return self.classifier(
            self.norm(h + self.residual(h))
        )

checkpoint = torch.load(
    ADVANCED / "advanced_residual_mlp.pt",
    map_location="cpu",
    weights_only=True
)

senses = checkpoint["senses"]

mlp = ResidualWSDClassifier(
    input_dim=768,
    hidden_dim=checkpoint["hidden_dim"],
    num_classes=len(senses)
)

mlp.load_state_dict(checkpoint["model_state"])
mlp.eval()

scaler = joblib.load(
    ADVANCED / "scaler.joblib"
)

print("Logistic Regression loaded.")
print("Residual MLP loaded.")
print("Classes:", senses)

Logistic Regression loaded.
Residual MLP loaded.
Classes: ['bank.1', 'bank.2', 'interest.1', 'interest.2', 'plant.1', 'plant.2', 'point.1', 'point.2', 'right.1', 'right.2', 'table.1', 'table.2']


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

words = [r["target_word"] for r in challenge_records]
y_true = np.array([
    r["sense_id"] for r in challenge_records
])

def restrict_scores(scores, classes, words):
    predictions = []

    for row_scores, word in zip(scores, words):
        allowed = [
            i for i, sense in enumerate(classes)
            if sense.startswith(word + ".")
        ]

        if len(allowed) != 2:
            raise ValueError(
                f"Expected two senses for {word}"
            )

        best = max(
            allowed,
            key=lambda i: row_scores[i]
        )

        predictions.append(classes[best])

    return np.array(predictions)


# Logistic Regression
lr_scores = logreg.decision_function(X_challenge)
lr_predictions = restrict_scores(
    lr_scores,
    list(logreg.classes_),
    words
)

# Residual MLP
X_scaled = scaler.transform(
    X_challenge
).astype(np.float32)

with torch.no_grad():
    mlp_scores = mlp(
        torch.tensor(X_scaled)
    ).numpy()

mlp_predictions = restrict_scores(
    mlp_scores,
    senses,
    words
)

results = {}

for name, pred in [
    ("Logistic Regression", lr_predictions),
    ("Residual MLP", mlp_predictions)
]:
    results[name] = {
        "accuracy": accuracy_score(y_true, pred),
        "macro_precision": precision_score(
            y_true, pred, labels=senses,
            average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, pred, labels=senses,
            average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, pred, labels=senses,
            average="macro", zero_division=0
        )
    }

    print(f"\n=== {name} ===")
    for metric, value in results[name].items():
        print(f"{metric}: {value:.4f}")

    print(classification_report(
        y_true, pred,
        labels=senses,
        zero_division=0
    ))


=== Logistic Regression ===
accuracy: 0.9167
macro_precision: 0.9458
macro_recall: 0.9167
macro_f1: 0.9089
              precision    recall  f1-score   support

      bank.1       1.00      1.00      1.00         3
      bank.2       1.00      1.00      1.00         3
  interest.1       1.00      1.00      1.00         3
  interest.2       1.00      1.00      1.00         3
     plant.1       1.00      0.67      0.80         3
     plant.2       0.75      1.00      0.86         3
     point.1       1.00      1.00      1.00         3
     point.2       1.00      1.00      1.00         3
     right.1       0.60      1.00      0.75         3
     right.2       1.00      0.33      0.50         3
     table.1       1.00      1.00      1.00         3
     table.2       1.00      1.00      1.00         3

    accuracy                           0.92        36
   macro avg       0.95      0.92      0.91        36
weighted avg       0.95      0.92      0.91        36


=== Residual MLP ===
acc

In [23]:
print("=== CHALLENGE ERROR ANALYSIS ===")

for i, row in enumerate(challenge_records):
    lr_ok = lr_predictions[i] == y_true[i]
    mlp_ok = mlp_predictions[i] == y_true[i]

    if not lr_ok or not mlp_ok:
        print("\nSentence:", row["text_en"])
        print("Gold:", y_true[i])
        print("LogReg:", lr_predictions[i])
        print("MLP:", mlp_predictions[i])

print("\nEvaluation complete.")

(OUT / "challenge_results.json").write_text(
    json.dumps(
        {
            "dataset": "assistant_reviewed_challenge_v1",
            "examples": len(challenge_records),
            "results": {
                name: {
                    key: float(value)
                    for key, value in metrics.items()
                }
                for name, metrics in results.items()
            }
        },
        indent=2
    ),
    encoding="utf-8"
)

print("Results saved.")

=== CHALLENGE ERROR ANALYSIS ===

Sentence: The plant requires sunlight and water.
Gold: plant.1
LogReg: plant.2
MLP: plant.2

Sentence: Turn right after the bridge.
Gold: right.2
LogReg: right.1
MLP: right.1

Sentence: Keep right at the intersection.
Gold: right.2
LogReg: right.1
MLP: right.1

Evaluation complete.
Results saved.
